In [22]:
import pandas as pd
import boto3
import io
import re

# --- Load CSV from S3 into memory ---
s3 = boto3.client('s3')
bucket = "sagemakerstack-transactionsrawdatabucket6643d2da-q7nfqsc3jpnn"
key = "hscode/HS.code2022.csv"

response = s3.get_object(Bucket=bucket, Key=key)
df = pd.read_csv(io.BytesIO(response["Body"].read()), header=None, dtype=str, low_memory=False)

# --- Define standard column names ---
final_columns = ['HSCode', 'Arabic_Description', 'English_Description', 'Unit', 'DutyRate']
cleaned_rows = []

# --- Regex pattern for valid HS codes (e.g., 01 02 03 04) ---
hs_pattern = re.compile(r"^\d{2}(?:\s?\d{2}){3}$")

for _, row in df.iterrows():
    row = row.dropna().astype(str).tolist()

    # Check for rows that contain a valid HSCode
    for i, val in enumerate(row):
        if hs_pattern.match(val.strip()):
            hs_code = val.replace(" ", "")
            arabic = row[i + 1] if i + 1 < len(row) else ""
            english = row[i + 2] if i + 2 < len(row) else ""
            unit = row[i + 3] if i + 3 < len(row) else ""
            duty = row[i + 4] if i + 4 < len(row) else ""

            # Handle "PROHIBITED" if it appears in any of the extracted fields
            if any("ممنوع" in field or "PROHIBITED" in field.upper() for field in [arabic, english, unit, duty]):
                unit = duty = "PROHIBITED"

            cleaned_rows.append([hs_code, arabic.strip(), english.strip(), unit.strip(), duty.strip()])
            break  # avoid capturing duplicate HSCode entries in one line

# --- Create DataFrame and save locally ---
print(f"✅ Cleaned file saved: hs_codes_cleaned.csv with {len(clean_df)} rows.")
clean_df = pd.DataFrame(cleaned_rows, columns=final_columns)
# Replace 'dash dash' patterns that trigger Excel with safer equivalents
clean_df['Arabic_Description'] = clean_df['Arabic_Description'].str.replace(r'^[-\s]+', '– ', regex=True)
clean_df['English_Description'] = clean_df['English_Description'].str.replace(r'^[-\s]+', '– ', regex=True)
clean_df.to_csv("cleaned_hscode.csv", index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL)

✅ Cleaned file saved: hs_codes_cleaned.csv with 7856 rows.
